In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

notebooks_folder = Path.cwd()
raw_folder = notebooks_folder.parent / "data" / "raw"
processed_folder = notebooks_folder.parent / "data" / "processed"


In [ ]:
datasets = {
    "aum_by_fund_house": "aum_by_fund_house.csv",
    "benchmark_indices": "benchmark_indices.csv",
    "category_inflows": "category_inflows.csv",
    "fund_master": "fund_master.csv",
    "industry_folio_count": "industry_folio_count.csv",
    "investor_transactions": "investor_transactions.csv",
    "monthly_sip_inflows": "monthly_sip_inflows.csv",
    "nav_history": "nav_history.csv",
    "portfolio_holdings": "portfolio_holdings.csv",
    "scheme_performance": "scheme_performance.csv"
}

In [ ]:
dataframes = {}

for name, file in datasets.items():
    df = pd.read_csv(raw_folder / file)
    dataframes[name] = df

In [ ]:
for name, df in dataframes.items():
    print(f"DATASET: {name.upper()}")
    print("="*80)

    print("\nShape:")
    print(df.shape)

    print("\nData Types:")
    print(df.dtypes)

    print("\nFirst 5 Rows:")
    display(df.head())

In [ ]:
for name, df in dataframes.items():
    print(f"MISSING VALUES - {name.upper()}")
    missing = df.isnull().sum()
    print("The number of missing values are ",missing)

In [ ]:
for name, df in dataframes.items():
    duplicates = df.duplicated().sum()
    print(f"{name}: {duplicates} duplicate rows")

In [ ]:
quality_report = []

for name, df in dataframes.items():
    quality_report.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isnull().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })

quality_df = pd.DataFrame(quality_report)

display(quality_df)

In [ ]:
fund_master = dataframes["fund_master"]

print("Total Schemes:", len(fund_master))
print()

print("Unique Fund Houses:")
print(fund_master["fund_house"].nunique())
print(fund_master["fund_house"].sort_values().unique())

print("\nUnique Categories:")
print(fund_master["category"].unique())

print("\nUnique Sub Categories:")
print(fund_master["sub_category"].unique())

print("\nUnique Risk Categories:")
print(fund_master["risk_category"].unique())

print("\nSEBI Category Codes:")
print(fund_master["sebi_category_code"].unique())

In [ ]:
print("AMFI Code Summary")

print("\nMin Code:")
print(fund_master["amfi_code"].min())

print("\nMax Code:")
print(fund_master["amfi_code"].max())

print("\nSample Codes:")
print(fund_master["amfi_code"].head(20).tolist())

In [ ]:
fund_master_codes = set(
    dataframes["fund_master"]["amfi_code"].astype(str)
)

nav_codes = set(
    dataframes["nav_history"]["amfi_code"].astype(str)
)

missing_in_nav = fund_master_codes - nav_codes

print("Fund Master Codes:", len(fund_master_codes))
print("NAV History Codes:", len(nav_codes))
print("Missing Codes:", len(missing_in_nav))

In [ ]:
missing_df = pd.DataFrame({
    "missing_amfi_code": list(missing_in_nav)
})

display(missing_df.head(20))

In [ ]:
print("DATA QUALITY SUMMARY")

total_codes = len(fund_master_codes)
matched_codes = total_codes - len(missing_in_nav)

coverage_pct = round((matched_codes / total_codes) * 100,2)

print(f"Total Fund Master AMFI Codes : {total_codes}")
print(f"Codes Found In NAV History   : {matched_codes}")
print(f"Missing Codes                : {len(missing_in_nav)}")
print(f"Coverage %                   : {coverage_pct}%")

if len(missing_in_nav) == 0:
    print("\nAll AMFI codes are available in NAV history.")
else:
    print(f"\n {len(missing_in_nav)} AMFI codes are missing from NAV history.")